In [20]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

file_path = r'C:\Users\sainz\Downloads\final_clean_merged_owid.xlsx'
df = pd.read_excel(file_path)

In [23]:
import altair as alt
import pandas as pd

alt.data_transformers.disable_max_rows()

df = df[df['year'] <= 2014].dropna(subset=['region', 'year'])

df_long = df.melt(
    id_vars=['year', 'region', 'country_x'],
    value_vars=['fossil_share_elec', 'renewables_share_elec'],
    var_name='energy_type',
    value_name='share'
).dropna(subset=['share'])

df_long['energy_type'] = df_long['energy_type'].replace({
    'fossil_share_elec': 'Fossil Fuel Share',
    'renewables_share_elec': 'Renewable Share'
})

# Scatter plot data prep
df_energy = df[['country_x','year','region',
                'co2_per_capita','child_mortality']].dropna()

region_select = alt.selection_point(
    fields=['region'],
    toggle='true', 
    clear='dblclick',  
    on='click'   
)

year_slider = alt.param(
    name='Year',
    value=2010,
    bind=alt.binding_range(min=int(df['year'].min()), max=2014, step=1)
)

dashboard_title = alt.Chart(
    pd.DataFrame({'text': ['Energy Mix vs Environmental & Health Impact']})
).mark_text(
    align='center',
    fontSize=28,
    fontWeight='bold'
).encode(text='text').properties(width=1000, height=40)

# Heatmap
heatmap = (
    alt.Chart(df_long)
    .transform_filter("datum.year == Year")
    .mark_rect()
    .encode(
        x=alt.X('region:N', sort='-y', title='Region'),
        y=alt.Y('energy_type:N', title='Energy Type'),
        color=alt.Color('share:Q', scale=alt.Scale(scheme='viridis'), title='% Share'),
        stroke=alt.condition(region_select, alt.value('black'), alt.value(None)),
        strokeWidth=alt.condition(region_select, alt.value(2), alt.value(0)),
        tooltip=[
            alt.Tooltip('country_x:N', title='Country'),
            alt.Tooltip('region:N', title='Region'),
            alt.Tooltip('energy_type:N', title='Energy Type'),
            alt.Tooltip('share:Q', title='% Share'),
            alt.Tooltip('year:O', title='Year')]
    )
    .add_params(region_select, year_slider)
    .properties(width=600, height=220,
                title="Fossil vs Renewable Electricity Share by Region")
)

# Scatter plot
scatter = (
    alt.Chart(df_energy)
    .transform_filter(region_select)
    .transform_filter("datum.year == Year")        
    .mark_circle(size=100)
    .encode(
        x=alt.X('co2_per_capita:Q', title='CO₂ Emissions per Capita'),
        y=alt.Y('child_mortality:Q', title='Child Mortality'),
        color=alt.Color('region:N', title='Region'),
        tooltip=[
            alt.Tooltip('country_x:N', title='Country'),
            alt.Tooltip('region:N', title='Region'),
            alt.Tooltip('year:O', title='Year'),
            alt.Tooltip('co2_per_capita:Q', title='CO₂ per Capita'),
            alt.Tooltip('child_mortality:Q', title='Child Mortality')
        ]
    )
    .add_params(region_select, year_slider)
    .properties(width=600, height=420,
                title="CO₂ Emissions vs Child Mortality")
)

# Final merge
dashboard = alt.vconcat(dashboard_title, heatmap, scatter).configure_title(
    fontSize=28, anchor='middle', offset=25
).configure_axis(
    titleFontSize=14
).configure_legend(
    titleFontSize=12, labelFontSize=11
)

dashboard

alt.VConcatChart(...)